# **FASE 1 - Construcción del dataset de ventanas**

In [1]:
# ==========================================
# 1. MONTAJE GOOGLE DRIVE
# ==========================================

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# ==========================================
# 2. LIBRERÍAS
# ==========================================

import os
import tarfile

import numpy as np
import pandas as pd

In [3]:
# ==========================================
# 3. CONFIGURACIÓN GENERAL
# ==========================================

RUTA_DATASET = "/content/drive/MyDrive/TFM/iot_23_datasets_small.tar.gz"

DESTINO = "/content/iot23"

VENTANAS_ESTUDIO = [1, 2, 5, 10, 30]

ESCENARIOS = {

    "Benigno_4_1": {
        "carpeta": "CTU-Honeypot-Capture-4-1",
        "tipo": "Benigno"
    },

    "Benigno_5_1": {
        "carpeta": "CTU-Honeypot-Capture-5-1",
        "tipo": "Benigno"
    },

    "Benigno_7_1": {
        "carpeta": "CTU-Honeypot-Capture-7-1/Somfy-01",
        "tipo": "Benigno"
    },

    "DDoS": {
        "carpeta": "CTU-IoT-Malware-Capture-48-1",
        "tipo": "DDoS"
    },

    "Scanning": {
        "carpeta": "CTU-IoT-Malware-Capture-3-1",
        "tipo": "Scanning"
    },

    "Botnet": {
        "carpeta": "CTU-IoT-Malware-Capture-1-1",
        "tipo": "Botnet"
    },

    "C&C": {
        "carpeta": "CTU-IoT-Malware-Capture-20-1",
        "tipo": "C&C"
    }

}

print("✅ Configuración cargada")

✅ Configuración cargada


In [4]:
# ==========================================
# 4. EXTRACCIÓN DE ESCENARIOS
# ==========================================

os.makedirs(
    DESTINO,
    exist_ok=True
)

ESCENARIOS_EXTRAER = [

    "CTU-Honeypot-Capture-4-1",
    "CTU-Honeypot-Capture-5-1",
    "CTU-Honeypot-Capture-7-1",

    "CTU-IoT-Malware-Capture-48-1",
    "CTU-IoT-Malware-Capture-3-1",
    "CTU-IoT-Malware-Capture-1-1",
    "CTU-IoT-Malware-Capture-20-1"

]

print("Extrayendo escenarios...")

with tarfile.open(
    RUTA_DATASET,
    "r:gz"
) as tar:

    miembros = []

    for member in tar.getmembers():

        if any(
            nombre in member.name
            for nombre in ESCENARIOS_EXTRAER
        ):
            miembros.append(member)

    tar.extractall(
        path=DESTINO,
        members=miembros
    )

print("✅ Escenarios extraídos")

Extrayendo escenarios...


/tmp/ipykernel_6339/4041635249.py:40: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(


✅ Escenarios extraídos


In [5]:
# ==========================================
# 5. DEFINICIÓN COLUMNAS ZEEK
# ==========================================

COLUMNAS_ZEEK = [

    'ts',
    'uid',
    'id.orig_h',
    'id.orig_p',
    'id.resp_h',
    'id.resp_p',
    'proto',
    'service',
    'duration',
    'orig_bytes',
    'resp_bytes',
    'conn_state',
    'local_orig',
    'local_resp',
    'missed_bytes',
    'history',
    'orig_pkts',
    'orig_ip_bytes',
    'resp_pkts',
    'resp_ip_bytes',
    'tunnel_parents',
    'label',
    'detailed-label'

]

In [6]:
# ==========================================
# 6. LECTURA DE ARCHIVOS conn.log.labeled
# ==========================================

def cargar_conn_log(ruta):

    # --------------------------------------
    # Leer cabecera real
    # --------------------------------------

    with open(
        ruta,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as f:

        for linea in f:

            if linea.startswith("#fields"):

                campos = (
                    linea
                    .replace("#fields", "")
                    .strip()
                    .split("\t")
                )

                break

    # --------------------------------------
    # Leer archivo
    # --------------------------------------

    df = pd.read_csv(
        ruta,
        sep="\t",
        comment="#",
        names=campos,
        engine="python"
    )

    # --------------------------------------
    # Separar la última columna
    # --------------------------------------

    ultima_col = (
        "tunnel_parents   label   detailed-label"
    )

    if ultima_col in df.columns:

        extra = (

            df[ultima_col]

            .astype(str)

            .str.split(
                expand=True
            )

        )

        df["tunnel_parents"] = extra[0]

        df["label"] = extra[1]

        df["detailed-label"] = extra[2]

        df.drop(
            columns=[ultima_col],
            inplace=True
        )

    return df

In [7]:
# ==========================================
# 7. CARGA DE ESCENARIOS
# ==========================================

dataframes_escenarios = {}

for nombre, info in ESCENARIOS.items():

    ruta = os.path.join(
        DESTINO,
        "opt",
        "Malware-Project",
        "BigDataset",
        "IoTScenarios",
        info["carpeta"],
        "bro",
        "conn.log.labeled"
    )

    print(f"Cargando {nombre}")

    df = cargar_conn_log(ruta)

    dataframes_escenarios[nombre] = df

    print(
        f"{nombre}: {len(df):,} flujos"
    )

Cargando Benigno_4_1
Benigno_4_1: 452 flujos
Cargando Benigno_5_1
Benigno_5_1: 1,374 flujos
Cargando Benigno_7_1
Benigno_7_1: 130 flujos
Cargando DDoS
DDoS: 3,394,338 flujos
Cargando Scanning
Scanning: 156,103 flujos
Cargando Botnet
Botnet: 1,008,748 flujos
Cargando C&C
C&C: 3,209 flujos


In [8]:
# ==========================================
# 8. LIMPIEZA - CONVERSIÓN DE TIPOS
# ==========================================

def limpiar_dataframe(df):

    columnas_numericas = [

        "ts",
        "duration",

        "orig_bytes",
        "resp_bytes",

        "orig_pkts",
        "resp_pkts"

    ]

    for col in columnas_numericas:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

    df = df.fillna(0)

    # flujo malicioso real
    df["flujo_malicioso"] = (

        df["label"]
        .astype(str)
        .str.contains(
            "Malicious",
            case=False,
            na=False
        )

    )

    return df

for nombre in dataframes_escenarios:

    dataframes_escenarios[nombre] = limpiar_dataframe(
        dataframes_escenarios[nombre]
    )

In [9]:
# ==========================================
# 9. INICIO ATAQUE POR ESCENARIO
# ==========================================

inicio_real_ataque = {}

for nombre, df in dataframes_escenarios.items():

    maliciosos = df[
        df["flujo_malicioso"]
    ]

    if len(maliciosos) > 0:

        inicio_real_ataque[nombre] = (
            maliciosos["ts"].min()
        )

    else:

        inicio_real_ataque[nombre] = np.nan

inicio_real_ataque

{'Benigno_4_1': nan,
 'Benigno_5_1': nan,
 'Benigno_7_1': nan,
 'DDoS': 1551377734.188184,
 'Scanning': 1526756261.8665,
 'Botnet': 1525879831.015811,
 'C&C': 1538479103.951732}

In [10]:
# ==========================================
# 10. ASIGNACIÓN DE VENTANAS
# ==========================================

def asignar_ventanas(
    df,
    tamaño_ventana
):

    ts_min = df["ts"].min()

    df["ventana_id"] = (

        (df["ts"] - ts_min)
        //
        tamaño_ventana

    ).astype(int)

    return df

In [11]:
# ==========================================
# 11. EXTRACCIÓN DE CARACTERÍSTICAS
# ==========================================

def extraer_caracteristicas_ventana(grupo):

    total_bytes = (
        grupo["orig_bytes"].sum()
        +
        grupo["resp_bytes"].sum()
    )

    total_pkts = (
        grupo["orig_pkts"].sum()
        +
        grupo["resp_pkts"].sum()
    )

    return pd.Series({

        "num_conexiones":
        len(grupo),

        "bytes_totales":
        total_bytes,

        "paquetes_totales":
        total_pkts,

        "ips_destino_unicas":
        grupo["id.resp_h"].nunique(),

        "puertos_destino_unicos":
        grupo["id.resp_p"].nunique(),

        "duracion_media":
        grupo["duration"].mean(),

        "duracion_max":
        grupo["duration"].max(),

        "orig_bytes":
        grupo["orig_bytes"].sum(),

        "resp_bytes":
        grupo["resp_bytes"].sum(),

        "orig_pkts":
        grupo["orig_pkts"].sum(),

        "resp_pkts":
        grupo["resp_pkts"].sum()

    })

In [12]:
# ==========================================
# 12. CREACIÓN DEL DATASET POR VENTANAS
# ==========================================

datasets_por_ventana = {}

for ventana in VENTANAS_ESTUDIO:

    print(f"\nProcesando ventanas de {ventana}s")

    lista_ventanas = []

    for escenario, df in dataframes_escenarios.items():

        print(f"  -> {escenario}")

        df_tmp = df.copy()

        df_tmp = asignar_ventanas(
            df_tmp,
            ventana
        )

        # --------------------------------------------------
        # Características
        # --------------------------------------------------

        features = (

            df_tmp

            .groupby("ventana_id")

            .apply(
                extraer_caracteristicas_ventana
            )

            .reset_index()

        )

        # --------------------------------------------------
        # Información temporal
        # --------------------------------------------------

        tiempos = (

            df_tmp

            .groupby("ventana_id")

            .agg(

                tiempo_inicio=(
                    "ts",
                    "min"
                ),

                tiempo_fin=(
                    "ts",
                    "max"
                )

            )

            .reset_index()

        )

        # --------------------------------------------------
        # Etiqueta real de la ventana
        # --------------------------------------------------

        labels = (

            df_tmp

            .groupby("ventana_id")

            ["flujo_malicioso"]

            .any()

            .astype(int)

            .reset_index(
                name="label_binaria"
            )

        )

        # --------------------------------------------------
        # Primer timestamp malicioso dentro de la ventana
        # --------------------------------------------------

        maliciosos = (

            df_tmp[
                df_tmp["flujo_malicioso"]
            ]

            .groupby("ventana_id")

            ["ts"]

            .min()

            .reset_index()

            .rename(
                columns={
                    "ts":
                    "primer_ts_malicioso_ventana"
                }
            )

        )

        # --------------------------------------------------
        # Merges
        # --------------------------------------------------

        features = features.merge(
            tiempos,
            on="ventana_id",
            how="left"
        )

        features = features.merge(
            labels,
            on="ventana_id",
            how="left"
        )

        features = features.merge(
            maliciosos,
            on="ventana_id",
            how="left"
        )

        # --------------------------------------------------
        # Metadatos
        # --------------------------------------------------

        features["escenario"] = escenario

        features["tipo_escenario"] = (
            ESCENARIOS[escenario]["tipo"]
        )

        features["ventana_global"] = range(
            len(features)
        )

        features[
            "inicio_real_ataque_escenario"
        ] = inicio_real_ataque[
            escenario
        ]

        lista_ventanas.append(
            features
        )

    dataset_final = pd.concat(
        lista_ventanas,
        ignore_index=True
    )

    datasets_por_ventana[
        ventana
    ] = dataset_final

    print(
        f"Dataset {ventana}s -> "
        f"{dataset_final.shape}"
    )


Procesando ventanas de 1s
  -> Benigno_4_1
  -> Benigno_5_1


/tmp/ipykernel_6339/1962829176.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/tmp/ipykernel_6339/1962829176.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/tmp/ipykernel_6339/1962829176.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the o

  -> Benigno_7_1
  -> DDoS


/tmp/ipykernel_6339/1962829176.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


  -> Scanning


...

/tmp/ipykernel_6339/1962829176.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


  -> Botnet


/tmp/ipykernel_6339/1962829176.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


  -> C&C
Dataset 30s -> (21247, 20)


/tmp/ipykernel_6339/1962829176.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [13]:
# ==========================================
# 13. DISTRIBUCIÓN DE CLASES
# ==========================================

for ventana, df in datasets_por_ventana.items():

    print("\n" + "="*60)

    print(
        f"VENTANA {ventana}s"
    )

    print("="*60)

    print(
        df["label_binaria"]
        .value_counts()
    )

    print()

    print(
        round(
            df["label_binaria"]
            .value_counts(
                normalize=True
            ) * 100,
            2
        )
    )


VENTANA 1s
label_binaria
1    449748
0     66501
Name: count, dtype: int64

label_binaria
1    87.12
0    12.88
Name: proportion, dtype: float64

VENTANA 2s
label_binaria
1    257887
0      9025
Name: count, dtype: int64

label_binaria
1    96.62
0     3.38
Name: proportion, dtype: float64

VENTANA 5s
label_binaria
1    110995
0      3502
Name: count, dtype: int64

label_binaria
1    96.94
0     3.06
Name: proportion, dtype: float64

VENTANA 10s
label_binaria
1    56645
0     3117
Name: count, dtype: int64

label_binaria
1    94.78
0     5.22
Name: proportion, dtype: float64

VENTANA 30s
label_binaria
1    18886
0     2361
Name: count, dtype: int64

label_binaria
1    88.89
0    11.11
Name: proportion, dtype: float64


In [14]:
# ==========================================
# 13.1 Prueba
# ==========================================

for ventana, df in datasets_por_ventana.items():

    print(
        f"\nVERIFICACIÓN VENTANA {ventana}s"
    )

    display(

        df[
            [

                "escenario",

                "tiempo_inicio",

                "tiempo_fin",

                "label_binaria",

                "primer_ts_malicioso_ventana",

                "inicio_real_ataque_escenario"

            ]

        ]

        .sample(
            10,
            random_state=42
        )

    )

    break


VERIFICACIÓN VENTANA 1s


,escenario,tiempo_inicio,tiempo_fin,label_binaria,primer_ts_malicioso_ventana,inicio_real_ataque_escenario
40036,Scanning,1.526766e+09,1.526766e+09,1,1.526766e+09,1.526756e+09
59897,Scanning,1.526801e+09,1.526801e+09,1,1.526801e+09,1.526756e+09
428652,Botnet,1.526198e+09,1.526198e+09,1,1.526198e+09,1.525880e+09
66828,Scanning,1.526811e+09,1.526811e+09,1,1.526811e+09,1.526756e+09
157618,Botnet,1.525926e+09,1.525926e+09,1,1.525926e+09,1.525880e+09
136086,Botnet,1.525905e+09,1.525905e+09,1,1.525905e+09,1.525880e+09
224898,Botnet,1.525994e+09,1.525994e+09,1,1.525994e+09,1.525880e+09
367027,Botnet,1.526136e+09,1.526136e+09,1,1.526136e+09,1.525880e+09
228779,Botnet,1.525998e+09,1.525998e+09,0,NaN,1.525880e+09
80208,Scanning,1.526833e+09,1.526833e+09,1,1.526833e+09,1.526756e+09


In [15]:
# ==========================================
# 14. ESCENARIOS CON Y SIN FASE BENIGNA
# ==========================================

resumen_escenarios = []

UMBRAL_FASE_PREVIA = 60  # segundos

for escenario, ts in inicio_real_ataque.items():

    if pd.isna(ts):

        tipo = "Benigno"

        retraso = np.nan

    else:

        df_tmp = dataframes_escenarios[
            escenario
        ]

        inicio_captura = (
            df_tmp["ts"].min()
        )

        retraso = (
            ts - inicio_captura
        )

        if retraso >= UMBRAL_FASE_PREVIA:

            tipo = (
                "Fase benigna previa"
            )

        else:

            tipo = (
                "Ataque desde inicio"
            )

    resumen_escenarios.append({

        "Escenario": escenario,

        "Retraso primer ataque (s)":
        retraso,

        "Tipo": tipo

    })

df_resumen_escenarios = pd.DataFrame(
    resumen_escenarios
)

df_resumen_escenarios

,Escenario,Retraso primer ataque (s),Tipo
0,Benigno_4_1,NaN,Benigno
1,Benigno_5_1,NaN,Benigno
2,Benigno_7_1,NaN,Benigno
3,DDoS,0.206654,Ataque desde inicio
4,Scanning,21.908494,Ataque desde inicio
5,Botnet,0.000738,Ataque desde inicio
6,C&C,334.351439,Fase benigna previa


In [16]:
# ==========================================
# 15. RESUMEN POR ESCENARIO Y VENTANA
# ==========================================

resumen_dataset = []

for ventana, df in datasets_por_ventana.items():

    for escenario in df["escenario"].unique():

        tmp = df[
            df["escenario"] == escenario
        ]

        resumen_dataset.append({

            "Ventana (s)": ventana,

            "Escenario": escenario,

            "Ventanas totales":
            len(tmp),

            "Ventanas benignas":
            (tmp["label_binaria"] == 0).sum(),

            "Ventanas maliciosas":
            (tmp["label_binaria"] == 1).sum()

        })

df_resumen_dataset = pd.DataFrame(
    resumen_dataset
)

df_resumen_dataset

,Ventana (s),Escenario,Ventanas totales,Ventanas benignas,Ventanas maliciosas
0,1,Benigno_4_1,246,246,0
1,1,Benigno_5_1,567,567,0
2,1,Benigno_7_1,92,92,0
3,1,DDoS,33612,163,33449
4,1,Scanning,76547,466,76081
5,1,Botnet,402261,62059,340202
6,1,C&C,2924,2908,16
7,2,Benigno_4_1,240,240,0
8,2,Benigno_5_1,528,528,0
9,2,Benigno_7_1,77,77,0


In [17]:
# ==========================================
# 16. ALMACENAMIENTO DE RESUMEN
# ==========================================

ruta_resumen = (
    "/content/drive/MyDrive/TFM/"
    "resumen_dataset_v2.xlsx"
)

df_resumen_dataset.to_excel(
    ruta_resumen,
    index=False
)

print(
    "✅ Tabla resumen guardada"
)

✅ Tabla resumen guardada


In [19]:
# ==========================================
# 17. GUARDAR DATASETS
# ==========================================

SALIDA = (
    "/content/drive/MyDrive/TFM/"
    "Datasets_Ventanas_V2"
)

os.makedirs(
    SALIDA,
    exist_ok=True
)

for tamaño, df in datasets_por_ventana.items():

    ruta = os.path.join(

        SALIDA,

        f"dataset_{tamaño}s.csv"

    )

    df.to_csv(
        ruta,
        index=False
    )

    print(
        f"✅ Guardado dataset_{tamaño}s.csv"
    )

✅ Guardado dataset_1s.csv
✅ Guardado dataset_2s.csv
✅ Guardado dataset_5s.csv
✅ Guardado dataset_10s.csv
✅ Guardado dataset_30s.csv
